# Runtime Verification and Baseline

I am your trainer in this notebook. We will move in tiny steps so learners can type with confidence.

**Outcome:** verify runtime, then capture one trustworthy baseline number.

## Step 1 - Confirm hardware

Pause and ask: **What GPU do you expect today?**

In [ ]:
!nvidia-smi

Expected output: a GPU model table. If this fails, switch Colab runtime to GPU first.

In [ ]:
import time
import torch

In [ ]:
print("CUDA available:", torch.cuda.is_available())

In [ ]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Step 2 - Build a tiny baseline experiment

Common mistake: timing GPU code without synchronization.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.randn((1024, 1024), device=device)
y = torch.randn((1024, 1024), device=device)

In [ ]:
if device == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()

In [ ]:
for _ in range(10):
    _ = torch.matmul(x, y)

In [ ]:
if device == "cuda":
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start
print(round(elapsed, 4))

## Checkpoint — Is this number reusable as a baseline?

**The question we asked:** Is this number reusable as a baseline? Why?

**Model answer (what a strong understanding looks like)**

Yes — with important caveats. The  timing we just captured is trustworthy *for this exact workload* (1024×1024 matmul) on *this exact GPU instance* under the conditions that existed in this Colab session, because we:

- Verified we were actually on a GPU runtime before measuring anything
- Used `torch.cuda.synchronize()` so the wall-clock timer only counted real GPU work, not host-side queuing
- Ran a modest number of iterations (10) to reduce noise from first-launch overhead

It is **not** a universal constant you can quote as "my GPU does X matmuls per second." Change the matrix size, switch to a different CUDA operation, move tensors on and off the device between calls, run on a different Colab GPU generation, or even hit a different time of day when Colab assigns different hardware, and the number will be different. Treat every serious experiment as requiring its own short calibration run under the exact conditions of that experiment.

**The deeper point:** A baseline is only meaningful when you can reproduce the measurement conditions. The real skill you practiced is the *preflight + synchronize + capture* ritual that makes later speedups or regressions believable instead of mysterious.

**If your answer went in a different direction, here is the key distinction most people miss the first time:**
They treat the number as a property of the *hardware*. It is actually a property of the *measurement setup*. Good engineers never compare two timings unless they can point to the controlled variables that make the comparison valid.

**Common misconception that feels very reasonable**

"I got 0.1234 seconds, therefore my GPU performs roughly 80 of these operations per second."

This feels intuitive but is usually wrong for real work. Real training or inference workloads include data loading, host-to-device copies, kernel launch overhead, memory allocation churn, mixed-precision effects, and entirely different operation mixes. The number you just measured is a *local calibration point*, not a hardware spec. The habit of always asking "under what conditions is this number true?" will save you from publishing nonsense benchmarks later in your career.

**If you're still unsure or the number on your screen looks nothing like what you expected**

That is extremely common and almost never means you did something wrong. The two most frequent causes:

1. You were accidentally on a CPU-only runtime (check the first cells again).
2. Colab gave you a different underlying GPU (T4 vs V100 vs A100, etc.). Your timing is still valid for *your* session — just note which GPU you actually got in your lab notes.

The important outcome is not that your number matches anyone else's. The important outcome is that you now know how to *produce* a number you can defend.


## Lesson Recap — What You Actually Learned

- You can reliably detect whether you are on a real GPU runtime before you waste time benchmarking the wrong thing.
- You understand why `torch.cuda.synchronize()` is non-negotiable for any timing that involves the GPU.
- You performed one controlled, defensible measurement that can serve as a local baseline for later comparisons in this course.
- You saw (and hopefully felt) the difference between "running code" and "getting trustworthy data from that run."

**A small human note:** If this felt like a lot of ceremony for a single number, you are experiencing exactly what professional GPU engineers feel every time they start a new performance investigation. The ceremony is what separates "I think the GPU made it faster" from "I can prove it and I can reproduce it." You just leveled up.


## Role Lens — Why This Matters in Real Work

**DevOps / MLOps** — Every time you get paged for "training is slow" or "inference latency spiked," the first 60 seconds of your investigation will be exactly this ritual: confirm the runtime, confirm the hardware, establish a credible baseline before you start changing things. Skipping it is how you end up chasing ghosts for three days.

**Data Science** — When you try a new model architecture or a bigger batch size and someone asks "is it actually faster or did we just get lucky with a warm cache?", you will need to point to a controlled before/after measurement done the way you just practiced. This is how you defend your tuning decisions in papers, reviews, and stand-ups.

**Data Engineering** — GPU-accelerated ETL pipelines (cuDF, Polars GPU, RAPIDS) live or die by whether the expensive data movement and kernel launches are actually paying off. You will run exactly these kinds of micro-benchmarks to decide whether moving a transform to the GPU is worth the PCIe transfer cost.

---

**You now have a trustworthy local baseline habit.**

Next section (GPU Fundamentals) we will use this exact skill to compare the same matrix multiplication on CPU vs GPU and start building real intuition about *why* the GPU wins on certain workloads. You are ready — the measurement foundation is solid.

Take a breath. The rest of the course builds on the honesty you just practiced.
